<a href="https://colab.research.google.com/github/ayush2459/APITesting_Python_RequestModule/blob/master/call_and_put.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np
import scipy
from scipy.stats import norm
from datetime import datetime, timedelta
import math
import logging
import json
import csv
import os

# ── LOGGING SETUP ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("pipeline.log", mode='w')
    ]
)
logger = logging.getLogger(__name__)

# ── CONSTANTS ─────────────────────────────────────────────────────────────────
R               = 0.05   # Default risk-free rate (5%)
TOLERANCE       = 1e-5   # IV convergence tolerance
MAX_ITERATIONS  = 15     # Newton-Raphson max iterations
IV_MIN          = 0.05   # 5%  minimum valid IV
IV_MAX          = 1.50   # 150% maximum valid IV
PARITY_EPSILON  = 0.50   # Max tolerable put-call parity error (currency units)
SYNC_MAX_DELTA_MS = 100  # Max allowed C/P timestamp desync in milliseconds


# ── HELPERS ───────────────────────────────────────────────────────────────────

def validate_tick_keys(tick: dict, required: list[str]) -> None:
    """Raise ValueError if any required key is missing or None in tick."""
    missing = [k for k in required if tick.get(k) is None]
    if missing:
        raise ValueError(f"Tick '{tick.get('asset_id', '?')}' missing fields: {missing}")


# ── FINANCIAL MODELS ──────────────────────────────────────────────────────────

class FinancialModels:
    """Implements Black-Scholes pricing and numerical implied volatility solver."""

    @staticmethod
    def _d1(S: float, K: float, T: float, R: float, sigma: float) -> float:
        if T <= 0 or sigma <= 0:
            return 0.0
        return (np.log(S / K) + (R + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))

    @staticmethod
    def _d2(S: float, K: float, T: float, R: float, sigma: float) -> float:
        return FinancialModels._d1(S, K, T, R, sigma) - sigma * np.sqrt(T)

    @staticmethod
    def black_scholes(S: float, K: float, T: float, R: float,
                      sigma: float, option_type: str = 'CALL') -> float:
        """
        Calculate theoretical option price using Black-Scholes formula.

        Args:
            S: Underlying asset price
            K: Strike price
            T: Time to expiry (in years)
            R: Risk-free interest rate
            sigma: Volatility (annualised)
            option_type: 'CALL' or 'PUT'

        Returns:
            Theoretical option price (float)
        """
        if T <= 0 or sigma <= 0:
            return max(0.0, S - K) if option_type == 'CALL' else max(0.0, K - S)

        d1 = FinancialModels._d1(S, K, T, R, sigma)
        d2 = FinancialModels._d2(S, K, T, R, sigma)

        if option_type == 'CALL':
            return S * norm.cdf(d1) - K * np.exp(-R * T) * norm.cdf(d2)
        elif option_type == 'PUT':
            return K * np.exp(-R * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        else:
            raise ValueError(f"Invalid option_type '{option_type}'. Must be 'CALL' or 'PUT'.")

    @staticmethod
    def calculate_vega(S: float, K: float, T: float, R: float, sigma: float) -> float:
        """Vega: sensitivity of option price to volatility."""
        if T <= 0 or sigma <= 0:
            return 0.0
        d1 = FinancialModels._d1(S, K, T, R, sigma)
        return S * norm.pdf(d1) * np.sqrt(T)

    @staticmethod
    def implied_volatility(market_price: float, S: float, K: float, T: float,
                           R: float, option_type: str = 'CALL',
                           initial_guess: float = 0.25) -> float:
        """
        Solve for implied volatility using Newton-Raphson iteration.

        Returns:
            IV (float) if converged within bounds, else -1.0
        """
        if market_price <= 0:
            return -1.0

        sigma = max(initial_guess, IV_MIN)  # ensure positive start

        for iteration in range(MAX_ITERATIONS):
            price_calc = FinancialModels.black_scholes(S, K, T, R, sigma, option_type)
            vega       = FinancialModels.calculate_vega(S, K, T, R, sigma)
            error      = market_price - price_calc

            if abs(error) < TOLERANCE:
                return sigma if IV_MIN <= sigma <= IV_MAX else -1.0

            if abs(vega) < 1e-8:
                logger.debug("Vega near zero at iteration %d — aborting IV solve.", iteration)
                break

            sigma -= error / vega

            # ── FIX: safe recovery when sigma goes non-positive ──
            if sigma <= 0:
                sigma = abs(sigma) * 0.5 + 0.1

        return -1.0


# ── VALIDATION ENGINE ─────────────────────────────────────────────────────────

class ValidationEngine:
    """Real-time data integrity checks on incoming market ticks."""

    SYNC_KEYS   = ['timestamp', 'put_timestamp', 'asset_id']
    PARITY_KEYS = ['underlying_price', 'strike', 'time_T', 'call_C', 'put_P']
    IV_KEYS     = ['underlying_price', 'strike', 'time_T', 'call_C', 'put_P']

    @staticmethod
    def check_synchronization(tick: dict) -> bool:
        """
        Flag tick as STALE if call/put timestamps are desynchronised
        beyond SYNC_MAX_DELTA_MS milliseconds.
        """
        try:
            validate_tick_keys(tick, ValidationEngine.SYNC_KEYS)
            ts_c = tick['timestamp']
            ts_p = tick['put_timestamp']

            if isinstance(ts_c, datetime) and isinstance(ts_p, datetime):
                delta_ms = abs((ts_c - ts_p).total_seconds() * 1000)
            else:
                delta_ms = abs(float(ts_c) - float(ts_p))

            if delta_ms > SYNC_MAX_DELTA_MS:
                tick['bug_flag']    = 'STALE'
                tick['error_detail'] = f"C/P sync delta: {delta_ms:.2f}ms (max {SYNC_MAX_DELTA_MS}ms)"
                logger.warning("[%s] STALE — %s", tick['asset_id'], tick['error_detail'])
                return True

            return False

        except ValueError as e:
            tick['bug_flag']    = 'ERROR'
            tick['error_detail'] = str(e)
            logger.error("[%s] Validation error: %s", tick.get('asset_id', '?'), e)
            return True
        except Exception as e:
            tick['bug_flag']    = 'ERROR'
            tick['error_detail'] = f"Unexpected: {e}"
            logger.exception("[%s] Unexpected sync error.", tick.get('asset_id', '?'))
            return True

    @staticmethod
    def check_parity(tick: dict) -> bool:
        """
        Flag tick as PARITY violation if |( C - P ) - ( S - K·e^{-rT} )| > PARITY_EPSILON.
        """
        try:
            validate_tick_keys(tick, ValidationEngine.PARITY_KEYS)
            S  = tick['underlying_price']
            K  = tick['strike']
            T  = tick['time_T']
            C  = tick['call_C']
            P  = tick['put_P']
            Rl = tick.get('risk_free_R', R)

            theoretical_forward_diff = S - K * np.exp(-Rl * T)
            observed_diff            = C - P
            parity_error             = abs(observed_diff - theoretical_forward_diff)
            tick['parity_error']     = round(parity_error, 6)

            if parity_error > PARITY_EPSILON:
                tick['bug_flag']     = 'PARITY'
                tick['error_detail'] = (f"Parity error: {parity_error:.4f} "
                                        f"(max {PARITY_EPSILON}) | "
                                        f"C-P={observed_diff:.4f}, TFD={theoretical_forward_diff:.4f}")
                logger.warning("[%s] PARITY — %s", tick['asset_id'], tick['error_detail'])
                return True

            return False

        except ValueError as e:
            tick['bug_flag']    = 'ERROR'
            tick['error_detail'] = str(e)
            logger.error("[%s] Parity check error: %s", tick.get('asset_id', '?'), e)
            return True

    @staticmethod
    def check_implied_volatility(tick: dict) -> bool:
        """
        Flag tick as IV_UNSTABLE if IV cannot be solved for call or put
        within valid bounds [IV_MIN, IV_MAX].
        """
        try:
            validate_tick_keys(tick, ValidationEngine.IV_KEYS)
            S  = tick['underlying_price']
            K  = tick['strike']
            T  = tick['time_T']
            C  = tick['call_C']
            P  = tick['put_P']
            Rl = tick.get('risk_free_R', R)
            ig = tick.get('historical_IV', 0.25)

            iv_c = FinancialModels.implied_volatility(C, S, K, T, Rl, 'CALL', ig)
            iv_p = FinancialModels.implied_volatility(P, S, K, T, Rl, 'PUT',  ig)

            tick['iv_c'] = round(iv_c, 6)
            tick['iv_p'] = round(iv_p, 6)

            if iv_c < 0 or iv_p < 0:
                tick['bug_flag']     = 'IV_UNSTABLE'
                tick['error_detail'] = f"IV solve failed — CALL_IV={iv_c:.3f}, PUT_IV={iv_p:.3f}"
                logger.warning("[%s] IV_UNSTABLE — %s", tick['asset_id'], tick['error_detail'])
                return True

            return False

        except ValueError as e:
            tick['bug_flag']    = 'ERROR'
            tick['error_detail'] = str(e)
            logger.error("[%s] IV check error: %s", tick.get('asset_id', '?'), e)
            return True


# ── BUG FIXER ─────────────────────────────────────────────────────────────────

class BugFixer:
    """Autonomous remediation logic for flagged ticks."""

    @staticmethod
    def fix_tick(tick: dict, last_validated_iv: float = 0.20) -> None:
        """
        Apply correction strategy based on bug_flag type.

        Strategies:
            STALE        → Recalculate C & P using last known good IV
            PARITY       → Symmetric adjustment to restore C - P = TFD
            IV_UNSTABLE  → Substitute last validated IV for C & P repricing
            ERROR        → Reject tick entirely
        """
        bug_flag = tick.get('bug_flag')
        asset    = tick.get('asset_id', '?')
        S  = tick.get('underlying_price')
        K  = tick.get('strike')
        T  = tick.get('time_T')
        Rl = tick.get('risk_free_R', R)

        if bug_flag == 'STALE':
            tick['call_C'] = round(FinancialModels.black_scholes(S, K, T, Rl, last_validated_iv, 'CALL'), 6)
            tick['put_P']  = round(FinancialModels.black_scholes(S, K, T, Rl, last_validated_iv, 'PUT'),  6)
            tick['fix_applied']       = 'STALE_RECALC'
            tick['iv_used_for_fix']   = last_validated_iv
            logger.info("[%s] Fixed STALE → repriced with IV=%.4f", asset, last_validated_iv)

        elif bug_flag == 'PARITY':
            tfd        = S - K * np.exp(-Rl * T)
            od         = tick['call_C'] - tick['put_P']
            adjustment = (od - tfd) / 2.0
            tick['call_C'] = round(max(0.0, tick['call_C'] - adjustment), 6)  # floor at 0 — options can't be negative
            tick['put_P']  = round(max(0.0, tick['put_P']  + adjustment), 6)  # floor at 0 — options can't be negative
            tick['fix_applied'] = 'PARITY_SYMMETRIC_ADJ'
            logger.info("[%s] Fixed PARITY → symmetric adj=%.6f", asset, adjustment)

        elif bug_flag == 'IV_UNSTABLE':
            tick['call_C'] = round(FinancialModels.black_scholes(S, K, T, Rl, last_validated_iv, 'CALL'), 6)
            tick['put_P']  = round(FinancialModels.black_scholes(S, K, T, Rl, last_validated_iv, 'PUT'),  6)
            tick['iv_c']   = last_validated_iv
            tick['iv_p']   = last_validated_iv
            tick['fix_applied'] = 'IV_SUBSTITUTION'
            logger.info("[%s] Fixed IV_UNSTABLE → substituted IV=%.4f", asset, last_validated_iv)

        elif bug_flag == 'ERROR':
            tick['fix_applied'] = 'REJECTED'
            tick['status']      = 'REJECTED'
            logger.error("[%s] Tick REJECTED — unrecoverable error.", asset)
            return

        # Mark as corrected and clean up flag
        if bug_flag in ('STALE', 'PARITY', 'IV_UNSTABLE'):
            tick['status'] = 'CORRECTED'
            tick.pop('bug_flag', None)
        else:
            tick['status'] = 'VALID'


# ── PIPELINE ──────────────────────────────────────────────────────────────────

def run_pipeline(ticks: list[dict]) -> list[dict]:
    """
    Process a batch of market ticks through the full validation + correction pipeline.

    Pipeline stages per tick:
        1. Synchronization check  → flag STALE
        2. Put-Call parity check  → flag PARITY
        3. Implied volatility check → flag IV_UNSTABLE
        4. Auto-remediation       → BugFixer.fix_tick()

    Args:
        ticks: List of tick dictionaries

    Returns:
        List of processed tick result dictionaries
    """
    results          = []
    last_valid_iv    = 0.20
    total            = len(ticks)
    valid_count      = 0
    corrected_count  = 0
    rejected_count   = 0

    logger.info("=" * 60)
    logger.info("Pipeline started — processing %d ticks", total)
    logger.info("=" * 60)

    for i, tick in enumerate(ticks, 1):
        logger.info("── Tick %d/%d  [%s] ──", i, total, tick.get('asset_id', '?'))

        # ── Stage 1: Synchronization ─────────────────────────────────────────
        flagged = ValidationEngine.check_synchronization(tick)

        # ── Stage 2: Parity (only if not already flagged) ────────────────────
        if not flagged:
            flagged = ValidationEngine.check_parity(tick)

        # ── Stage 3: Implied Volatility ───────────────────────────────────────
        if not flagged:
            flagged = ValidationEngine.check_implied_volatility(tick)

        # ── Stage 4: Remediation ──────────────────────────────────────────────
        if flagged:
            BugFixer.fix_tick(tick, last_valid_iv)
        else:
            # Update rolling last-known-good IV from valid tick
            iv_avg = (tick.get('iv_c', last_valid_iv) + tick.get('iv_p', last_valid_iv)) / 2.0
            last_valid_iv = round(iv_avg, 6)
            tick['status'] = 'VALID'

        # ── Tally ─────────────────────────────────────────────────────────────
        status = tick.get('status', 'VALID')
        if status == 'VALID':
            valid_count += 1
        elif status == 'CORRECTED':
            corrected_count += 1
        elif status == 'REJECTED':
            rejected_count += 1

        results.append(format_tick(tick))

    # ── Summary ───────────────────────────────────────────────────────────────
    logger.info("=" * 60)
    logger.info("Pipeline complete.")
    logger.info("  VALID:     %d / %d", valid_count,     total)
    logger.info("  CORRECTED: %d / %d", corrected_count, total)
    logger.info("  REJECTED:  %d / %d", rejected_count,  total)
    logger.info("  Final rolling IV: %.4f", last_valid_iv)
    logger.info("=" * 60)

    return results


# ── UTILITY ───────────────────────────────────────────────────────────────────

def format_tick(tick: dict) -> dict:
    """Return a clean summary dict for reporting."""
    return {
        'Asset':       tick.get('asset_id'),
        'S':           tick.get('underlying_price'),
        'K':           tick.get('strike'),
        'C_New':       tick.get('call_C'),
        'P_New':       tick.get('put_P'),
        'IV_Call':     tick.get('iv_c', 'N/A'),
        'IV_Put':      tick.get('iv_p', 'N/A'),
        'Parity_Err':  f"{tick.get('parity_error', 0):.4f}",
        'Status':      tick.get('status', 'VALID'),
        'Correction':  tick.get('fix_applied', 'N/A'),
        'Error_Detail': tick.get('error_detail', ''),
    }


def export_results(results: list[dict], filepath: str = "pipeline_output.csv") -> None:
    """Export pipeline results to a CSV file."""
    if not results:
        logger.warning("No results to export.")
        return
    with open(filepath, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    logger.info("Results exported to '%s'", filepath)


def load_ticks_from_json(filepath: str) -> list[dict]:
    """Load tick data from a JSON file."""
    with open(filepath, 'r') as f:
        ticks = json.load(f)
    logger.info("Loaded %d ticks from '%s'", len(ticks), filepath)
    return ticks


# ── GOOGLE COLAB HELPER ───────────────────────────────────────────────────────

def colab_export(csv_path: str = "pipeline_output.csv") -> None:
    """
    Call this after main() in Google Colab to:
      1. Display results as a table
      2. Download CSV to your computer
      3. Save CSV to Google Drive (permanent)

    Usage in Colab:
        main()
        colab_export()
    """
    try:
        import google.colab
    except ImportError:
        print("ℹ️  Not running in Google Colab — use the CSV file directly.")
        return

    import pandas as pd
    from IPython.display import display
    from google.colab import files

    # 1. Display table
    print("\n📋 RESULTS TABLE:")
    df = pd.read_csv(csv_path)
    display(df)

    # 2. Download to computer
    files.download(csv_path)
    print("⬇️  CSV download started to your computer.")

    # 3. Save to Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        import shutil
        drive_path = '/content/drive/MyDrive/' + csv_path
        shutil.copy(csv_path, drive_path)
        print(f"☁️  Saved to Google Drive → {drive_path}")
    except Exception as e:
        print(f"ℹ️  Drive save skipped: {e}")


# ── DEMO / TEST ───────────────────────────────────────────────────────────────

def main():
    """
    Demo: runs a batch of 5 sample ticks through the full pipeline
    covering all bug scenarios: VALID, STALE, PARITY, IV_UNSTABLE, ERROR.
    """
    now = datetime.now()

    sample_ticks = [
        # 1. VALID tick — clean data, should pass all checks
        {
            'asset_id':         'AAPL',
            'timestamp':        now,
            'put_timestamp':    now - timedelta(milliseconds=10),
            'underlying_price': 150.0,
            'strike':           150.0,
            'time_T':           0.25,
            'call_C':           7.97,
            'put_P':            6.49,
            'risk_free_R':      0.05,
            'historical_IV':    0.25,
        },
        # 2. STALE tick — put timestamp is 200ms behind call
        {
            'asset_id':         'TSLA',
            'timestamp':        now,
            'put_timestamp':    now - timedelta(milliseconds=200),
            'underlying_price': 250.0,
            'strike':           255.0,
            'time_T':           0.50,
            'call_C':           18.0,
            'put_P':            20.0,
            'risk_free_R':      0.05,
            'historical_IV':    0.30,
        },
        # 3. PARITY violation — C - P deviates too far from S - K*e^{-rT}
        {
            'asset_id':         'MSFT',
            'timestamp':        now,
            'put_timestamp':    now - timedelta(milliseconds=5),
            'underlying_price': 300.0,
            'strike':           300.0,
            'time_T':           0.25,
            'call_C':           15.0,   # inflated call → parity breaks
            'put_P':            8.0,
            'risk_free_R':      0.05,
            'historical_IV':    0.22,
        },
        # 4. IV_UNSTABLE — market prices are unrealistic for IV solve
        {
            'asset_id':         'GOOG',
            'timestamp':        now,
            'put_timestamp':    now - timedelta(milliseconds=20),
            'underlying_price': 100.0,
            'strike':           200.0,   # deep OTM — IV solve likely fails
            'time_T':           0.01,
            'call_C':           0.0001,
            'put_P':            0.0001,
            'risk_free_R':      0.05,
            'historical_IV':    0.20,
        },
        # 5. ERROR tick — missing required fields
        {
            'asset_id':         'AMZN',
            'timestamp':        now,
            # put_timestamp intentionally missing → ERROR
        },
    ]

    print("\n" + "=" * 60)
    print("   AlphaMarket Real-Time Bug Detection & Correction Engine")
    print("=" * 60 + "\n")

    results = run_pipeline(sample_ticks)

    print("\n📊 PIPELINE RESULTS:")
    print("-" * 60)
    for r in results:
        status_icon = {"VALID": "✅", "CORRECTED": "🔧", "REJECTED": "❌"}.get(r['Status'], "❓")
        print(f"{status_icon} [{r['Asset']}]  Status: {r['Status']:10s} | "
              f"C={r['C_New']}  P={r['P_New']}  "
              f"Parity Err={r['Parity_Err']}  "
              f"Fix={r['Correction']}")
        if r['Error_Detail']:
            print(f"   ⚠️  {r['Error_Detail']}")

    export_results(results, "pipeline_output.csv")
    print("\n✅ Results saved to pipeline_output.csv")
    print("📄 Full logs saved to pipeline.log\n")

    # auto-trigger Colab export
    colab_export()


if __name__ == "__main__":
    main()

ERROR:__main__:[AMZN] Validation error: Tick 'AMZN' missing fields: ['put_timestamp']
ERROR:__main__:[AMZN] Tick REJECTED — unrecoverable error.



   AlphaMarket Real-Time Bug Detection & Correction Engine


📊 PIPELINE RESULTS:
------------------------------------------------------------
🔧 [AAPL]  Status: CORRECTED  | C=6.922496  P=5.059166  Parity Err=0.3833  Fix=IV_SUBSTITUTION
   ⚠️  IV solve failed — CALL_IV=-1.000, PUT_IV=-1.000
🔧 [TSLA]  Status: CORRECTED  | C=14.713928  P=13.417956  Parity Err=0.0000  Fix=STALE_RECALC
   ⚠️  C/P sync delta: 200.00ms (max 100ms)
🔧 [MSFT]  Status: CORRECTED  | C=13.36333  P=9.63667  Parity Err=3.2733  Fix=PARITY_SYMMETRIC_ADJ
   ⚠️  Parity error: 3.2733 (max 0.5) | C-P=7.0000, TFD=3.7267
🔧 [GOOG]  Status: CORRECTED  | C=0.0  P=49.950112  Parity Err=99.9000  Fix=PARITY_SYMMETRIC_ADJ
   ⚠️  Parity error: 99.9000 (max 0.5) | C-P=0.0000, TFD=-99.9000
❌ [AMZN]  Status: REJECTED   | C=None  P=None  Parity Err=0.0000  Fix=REJECTED
   ⚠️  Tick 'AMZN' missing fields: ['put_timestamp']

✅ Results saved to pipeline_output.csv
📄 Full logs saved to pipeline.log


📋 RESULTS TABLE:


,Asset,S,K,C_New,P_New,IV_Call,IV_Put,Parity_Err,Status,Correction,Error_Detail
0,AAPL,150.0,150.0,6.922496,5.059166,0.2,0.2,0.3833,CORRECTED,IV_SUBSTITUTION,"IV solve failed — CALL_IV=-1.000, PUT_IV=-1.000"
1,TSLA,250.0,255.0,14.713928,13.417956,NaN,NaN,0.0000,CORRECTED,STALE_RECALC,C/P sync delta: 200.00ms (max 100ms)
2,MSFT,300.0,300.0,13.363330,9.636670,NaN,NaN,3.2733,CORRECTED,PARITY_SYMMETRIC_ADJ,"Parity error: 3.2733 (max 0.5) | C-P=7.0000, T..."
3,GOOG,100.0,200.0,0.000000,49.950112,NaN,NaN,99.9000,CORRECTED,PARITY_SYMMETRIC_ADJ,"Parity error: 99.9000 (max 0.5) | C-P=0.0000, ..."
4,AMZN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,REJECTED,REJECTED,Tick 'AMZN' missing fields: ['put_timestamp']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  CSV download started to your computer.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
☁️  Saved to Google Drive → /content/drive/MyDrive/pipeline_output.csv
